# 🚗 DoorDash Delivery Prediction
## Notebook 02 — Data Cleaning

**Goal:** Fix nulls, wrong data types, outliers, and duplicates. Save clean data to `data/processed/`.

**Author:** Divyargarg

---

In [1]:
# ── CELL 1: Import Libraries ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

print('✅ Libraries loaded!')

✅ Libraries loaded!


In [ ]:
# ── CELL 2: Load Raw Data ─────────────────────────────────────────────
df = pd.read_csv('../data/raw/historical_data.csv')

print(f'✅ Raw data loaded!')
print(f'📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# ── CELL 3: Snapshot Before Cleaning ─────────────────────────────────
# Always record state before making changes
rows_before = len(df)
cols_before = len(df.columns)
nulls_before = df.isnull().sum().sum()

print(f'📊 BEFORE CLEANING:')
print(f'   Rows    : {rows_before}')
print(f'   Columns : {cols_before}')
print(f'   Nulls   : {nulls_before}')

In [ ]:
# ── CELL 4: Remove Duplicate Rows ────────────────────────────────────
dupes = df.duplicated().sum()
print(f'🔁 Duplicate rows found: {dupes}')

df = df.drop_duplicates()
print(f'✅ Duplicates removed. Rows remaining: {len(df)}')

In [ ]:
# ── CELL 5: Inspect & Fix Data Types ─────────────────────────────────
print('📋 Current Data Types:')
print(df.dtypes)

# Fix timestamp columns if they exist — adjust column names to match yours
# Example: if 'created_at' is a datetime stored as string, convert it
timestamp_cols = [col for col in df.columns if 'time' in col.lower() or 'date' in col.lower()]
print(f'\n🕐 Potential timestamp columns: {timestamp_cols}')

for col in timestamp_cols:
    try:
        df[col] = pd.to_datetime(df[col])
        print(f'  ✅ Converted {col} to datetime')
    except Exception as e:
        print(f'  ⚠️  Could not convert {col}: {e}')

In [ ]:
# ── CELL 6: Handle Missing Values ────────────────────────────────────
print('📊 Missing Values Summary:')
null_df = pd.DataFrame({
    'nulls': df.isnull().sum(),
    'pct_missing': (df.isnull().sum() / len(df) * 100).round(2)
})
null_df = null_df[null_df['nulls'] > 0].sort_values('pct_missing', ascending=False)
print(null_df)

# Strategy per column — adjust based on your EDA findings:
# - Drop columns with > 50% missing
# - Impute numeric columns with median (safer than mean for skewed data)
# - Impute categorical columns with mode

# Drop columns with too many nulls
threshold = 0.5
cols_to_drop = null_df[null_df['pct_missing'] > threshold * 100].index.tolist()
if cols_to_drop:
    print(f'\n🗑️  Dropping columns with >50% nulls: {cols_to_drop}')
    df = df.drop(columns=cols_to_drop)
else:
    print('\n✅ No columns exceed 50% missing threshold')

In [ ]:
# ── CELL 7: Impute Remaining Nulls ───────────────────────────────────
# Numeric → median imputation
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f'  ✅ {col}: filled {df[col].isnull().sum()} nulls with median ({median_val:.2f})')

# Categorical → mode imputation
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f'  ✅ {col}: filled nulls with mode ("{mode_val}")')

print(f'\n📊 Total nulls remaining: {df.isnull().sum().sum()}')

In [ ]:
# ── CELL 8: Detect & Handle Outliers (IQR Method) ────────────────────
# IQR = Interquartile Range — values beyond 1.5x IQR are outliers

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

outlier_summary = []
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = (outliers / len(df)) * 100
    outlier_summary.append({'column': col, 'outliers': outliers, 'pct': pct, 'lower': lower, 'upper': upper})

outlier_df = pd.DataFrame(outlier_summary).sort_values('pct', ascending=False)
print('📊 Outlier Summary (IQR Method):')
print(outlier_df.to_string(index=False))

In [ ]:
# ── CELL 9: Cap Outliers (Winsorizing) ───────────────────────────────
# Instead of removing rows, we cap extreme values at the IQR boundaries
# This keeps the data but removes extreme distortion

for _, row in outlier_df.iterrows():
    col = row['column']
    if row['pct'] > 1:  # Only cap if more than 1% are outliers
        df[col] = df[col].clip(lower=row['lower'], upper=row['upper'])
        print(f'  ✅ Capped outliers in: {col} (range: {row["lower"]:.2f} → {row["upper"]:.2f})')

print('\n✅ Outlier capping complete!')

In [ ]:
# ── CELL 10: Boxplots After Cleaning ─────────────────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
n = len(numeric_cols)
cols = 3
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='#FF6600', alpha=0.6))
    axes[i].set_title(col, fontsize=11, fontweight='bold')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplots After Cleaning', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/02_boxplots_clean.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved to outputs/figures/')

In [ ]:
# ── CELL 11: Cleaning Summary ─────────────────────────────────────────
rows_after = len(df)
cols_after = len(df.columns)
nulls_after = df.isnull().sum().sum()

print('📊 CLEANING SUMMARY')
print(f"{'Metric':<20} {'Before':<15} {'After'}")
print('-' * 45)
print(f"{'Rows':<20} {rows_before:<15} {rows_after}")
print(f"{'Columns':<20} {cols_before:<15} {cols_after}")
print(f"{'Nulls':<20} {nulls_before:<15} {nulls_after}")

In [ ]:
# ── CELL 12: Save Clean Data ──────────────────────────────────────────
df.to_csv('../data/processed/cleaned_data.csv', index=False)
print('✅ Clean data saved to data/processed/cleaned_data.csv')
print(f'📊 Final shape: {df.shape}')

## 📝 Cleaning Decisions Log
*(Fill this in as you make decisions — copy to docs/decisions.md too)*

- **Duplicates:** 
- **Columns dropped:** 
- **Null strategy used:** 
- **Outlier handling:** 
- **Data type fixes:** 
- **Anything surprising:** 